# Get to Know a Dataset: human_brain_snm3C-seq

This notebook serves as a guided tour of the human_brain_snm3C-seq(https://registry.opendata.aws/salk-human-brain-snm3C-seq) dataset. More usage examples, tutorials, and documentation for this dataset and others can be found at the [Registry of Open Data on AWS](https://registry.opendata.aws/).

### Q: How have you organized your dataset? Help us understand the key prefix structure of your S3 bucket.

The dataset is organized by the following folder:
- single_cell_fastq: raw single-cell level fastq files
- WGS: raw Whole Genome Sequencing fastq
- allc: single-cell and pseudobulk allc.tsv.gz files
- hic: pseudobulk .hic files
- mcool: mcool format for pseudobulk 3D genome contact
- loop.Q.cool: imputed 3D genome contact cool files at 10kb resolution
- h5ad: .h5ad adata for gene and 100-kb mCG and mCH, and 100- and 25-kb 3D genome embedding
- ABC: the ABC model predicted enhancer-promoter links

### Q: What data formats are present in your dataset? What kinds of data are stored using these formats? Can you give any advice for how you work with these data formats?

#### File formats:
- (1) fastq format: [cell_id]_R1.fastq.gz and [cell_id]_R2.fastq.gz and whole genome sequencing fastq for each donor
- (2) .tsv.gz format: See ALLCools(https://github.com/lhqing/ALLCools) for allc format
- (3) .hic format: hic files are generated by juicebox (https://github.com/aidenlab/Juicebox).
- (4) .cool format: .cool format is used for cooler (https://github.com/open2c/cooler), cooltools (https://github.com/open2c/cooltools) and schicluster (https://github.com/zhoujt1994/scHiCluster)
- (5) .h5ad format: format for single cell data analysis using scanpy (https://github.com/scverse/scanpy)

### Q: Can you show us an example of downloading and loading data from your dataset?

See https://github.com/DingWB/BG_snm3C-seq for example

### Q: How to normalize the raw methylation fraction?

```python
import os,sys
import pandas as pd
import anndata

adata_path="BG.gene-CHN.h5ad" # example adata path
gene="DRD2" # an example gene
clip_norm_value=10
raw_adata = anndata.read_h5ad(os.path.expanduser(adata_path), backed='r')
adata = raw_adata[:, gene].to_memory()
raw_adata.file.close()
cols = adata.obs.columns.tolist()
if 'prior_mean' in cols:
    na_sum = adata.to_df().isna().sum().sum()
    adata.X = adata.X / adata.obs.prior_mean.values[:, None]  # range = [0,1,10]
    if not clip_norm_value is None:
        if issparse(adata.X):
            X=adata.X.toarray()
        else:
            X=adata.X
        adata.X = np.clip(X, None, clip_norm_value)
    adata.uns['normalize_per_cell'] = True
adata
```

### Q: cell id in metadata does not match the cell id from the mapping pipeline

The Broad mapping pipeline generates cell IDs slightly differently. Please use the following code to convert the cell IDs from the Broad pipeline into new_cell_id:
```python
import pandas as pd
def _parse_cell_id_v2(cell_id):
	plate, multiplex_group, pcr_index, random_index = cell_id.split('-')
	# 384 pos
	col384 = int(random_index[1:]) - 1
	row384 = ord(random_index[0]) - 65  # convert A-P to 0-23
	resolved_multiplex_group=(col384 % 12) // 2 + 1 if 'unknow' not in random_index.lower() else 'NA'
	uid=plate+'-'+str(resolved_multiplex_group)+'-'+pcr_index
	new_cell_id = uid + '-' + random_index
	record = pd.Series({
		'new_cell_id':new_cell_id,
		'Plate': plate,
		'PCRIndex': pcr_index,
		'MultiplexGroup': multiplex_group,
		'ResolvedMultiplexGroup': resolved_multiplex_group,
		'RandomIndex': random_index,
		'Col384': col384,
		'Row384': row384
	})
	return record

metadata=pd.read_csv("annotations.tsv",sep='\t',index_col=0)
cell_ids=metadata.index.tolist()
plate_info = pd.DataFrame([_parse_cell_id_v2(cell_id) for cell_id in cell_ids],index=cell_ids)
# use new_cell_id
```

### Q: What is one unanswered question that you think could be answered using these data? Do you have any recommendations or advice for someone wanting to answer this question?

How does cell-type-specific DNA methylation work together with 3D genome organization to regulate gene expression in human basal ganglia cell types, and how do disease-associated genetic variants act through this regulatory layer? Because snm3C-seq measures DNA methylation and chromatin conformation in the same single nucleus, this dataset makes it possible to study how these two layers relate within individual cells, rather than only across cell populations. That in turn could help identify which regulatory elements and target genes mediate genetic risk for disorders such as Parkinson’s disease, schizophrenia, and addiction in specific basal ganglia cell types.

Recommendations or advice for someone wanting to answer this question:

Start from our cell-type annotations, which are aligned to the BICAN/HMBA basal ganglia consensus taxonomy. This lets you integrate the data directly with companion RNA-seq, ATAC-seq, histone modification, and spatial datasets from the same consortium.
Aggregate cells by cell type (pseudobulk) for analyses that need high coverage, such as loop calling or region-level methylation. Single-cell 3D genome data are sparse.
Use non-CG methylation (mCH) for neurons and CG methylation (mCG) for non-neuronal cells, since mCH is a strong marker of neuronal gene activity.
Account for donor effects. The data come from seven donors of different ages and sexes.
Treat predicted enhancer–gene links as hypotheses, and validate them with functional experiments such as CRISPR perturbation or reporter assays.
Our analysis code (https://github.com/DingWB/BG_snm3C-seq) and companion papers in the BICAN basal ganglia package are good starting points.

For more detail of the usage of these data, please refer to our paper: 

BioRxiv: https://www.biorxiv.org/content/10.64898/2026.02.12.705594v2.full

Cell DOI: 10.1016/j.cell.2026.10.019

### Q: How to download data from AWS?

```bash
# Downloading a Folder from AWS S3

## 1. Getting Help for the AWS CLI
aws s3 help

# Help for a specific command
aws s3 ls help
aws s3 cp help
aws s3 sync help


## 2. Download an Entire Folder (Recommended: `aws s3 sync`)
sync only transfers missing or changed files, so re-running it after an interruption resumes the download.

aws s3 sync --no-sign-request s3://salk-human-brain-snm3c-seq/BasalGanglia/allc/SingleCell/ ./SingleCell/


### Check before downloading

# Show total size and number of objects
aws s3 ls --no-sign-request s3://salk-human-brain-snm3c-seq/BasalGanglia/allc/SingleCell/ \
    --recursive --human-readable --summarize | tail -2

# Dry run: list what would be downloaded without downloading
aws s3 sync s3://salk-human-brain-snm3c-seq/BasalGanglia/allc/SingleCell/ ./SingleCell/ --dryrun | head

### Useful options
# Public bucket and no credentials configured: skip request signing
aws s3 sync s3://.../SingleCell/ ./SingleCell/ --no-sign-request

# Download only allc files and their tabix indexes
aws s3 sync s3://salk-human-brain-snm3c-seq/BasalGanglia/allc/SingleCell/ ./SingleCell/ --no-sign-request \
    --exclude "*" --include "*.allc.tsv.gz" --include "*.allc.tsv.gz.tbi"

# Many small files: raise concurrency (default is 10)
aws configure set default.s3.max_concurrent_requests 64

## 3. Very Many Files: Use `s5cmd` (much faster)
[s5cmd](https://github.com/peak/s5cmd) is usually several to over ten times faster than the AWS CLI for large numbers of files.

# Install
conda install -c conda-forge s5cmd

# Download with 128 parallel workers
s5cmd --numworkers 128 cp 's3://salk-human-brain-snm3c-seq/BasalGanglia/allc/SingleCell/*' ./SingleCell/

# Public bucket: skip request signing
s5cmd --no-sign-request --numworkers 128 cp 's3://.../SingleCell/*' ./SingleCell/

# Resume: skip files whose size already matches
s5cmd --numworkers 128 cp --if-size-differ 's3://.../SingleCell/*' ./SingleCell/

## 4. Running on HPC
- Run long downloads inside `screen` or `tmux`, or submit them as a batch job, so an SSH disconnect does not kill the transfer.
- If a download is interrupted, re-run `aws s3 sync` (or `s5cmd cp --if-size-differ`) to resume.
```